In [ ]:
# Preprocesamiento de datos
Proyecto MIA - Preparación de métricas para detección de anomalías
#Librerias
import pandas as pd
import numpy as np
#Carga de datos
df = pd.read_csv("../data/sample/ejemplo_dataset.csv", sep=";")
df.head()
#Conversion de tipos
df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"], utc=True, errors="coerce")
df["value"] = pd.to_numeric(df["value"], errors="coerce")
#Eliminacion de nulos
df = df.dropna(subset=["timestamp_utc", "value", "instance", "metric"])
df.head()
#Buckets de 15 minutos
df["t15"] = df["timestamp_utc"].dt.floor("15min")
df.head()
#Agrupacion de metricas por intervalo
interval_df = (
    df.groupby(["t15", "instance", "metric"], as_index=False)["value"]
      .mean()
)
interval_df.head()
#Transformacion a formato ancho
wide = interval_df.pivot_table(
    index=["t15", "instance"],
    columns="metric",
    values="value",
    aggfunc="mean"
).reset_index()

wide = wide.sort_values(["instance", "t15"]).reset_index(drop=True)
wide.head()
#Continuidad temporal
full_data = []
for srv in wide["instance"].unique():
    temp = wide[wide["instance"] == srv].copy()
    full_range = pd.date_range(temp["t15"].min(), temp["t15"].max(), freq="15min")
    grid = pd.DataFrame({"t15": full_range, "instance": srv})
    temp_full = grid.merge(temp, on=["t15", "instance"], how="left")
    full_data.append(temp_full)

wide2 = pd.concat(full_data, ignore_index=True)
wide2 = wide2.sort_values(["instance", "t15"]).reset_index(drop=True)
wide2.head()
#Imputacion simple
id_cols = ["t15", "instance"]
feature_cols = [c for c in wide2.columns if c not in id_cols]

for col in feature_cols:
    wide2[col] = pd.to_numeric(wide2[col], errors="coerce")

wide2[feature_cols] = wide2[feature_cols].fillna(wide2[feature_cols].median(numeric_only=True))
wide2.head()


## Resultado del preprocesamiento

En esta etapa, los datos fueron limpiados, convertidos a un formato temporal homogéneo de 15 minutos y reorganizados en una estructura tabular adecuada para el entrenamiento del modelo. Además, se trataron los valores faltantes para mantener consistencia en las series temporales.
